# Índice de Problemas de Convivencia (IPC): construcción y resultados
**EVPI 2023-2024 vs. 2025-2026 · OMSCGR**

Este cuaderno reproduce el informe técnico del IPC y el Excel `EVPI_indice_problemas_convivencia.xlsx`.

1. **Covariables.** Armonización de las variables explicativas, con calles sin iluminación (CC_7) como explicativa.
2. **Ítems.** Codificación de la sección de convivencia (CC_1 a CC_6, CC_8 y CC_9).
3. **Validación psicométrica.** Correlaciones tetracóricas, factor dominante, α ordinal, KR-20, IRT 2PL y funcionamiento diferencial (DIF) por ola.
4. **Índice.** Versión principal, subíndices y versiones alternativas.
5. **Cambio entre olas.** Medias, distribución y subgrupos.
6. **Determinantes.** Modelos anidados, modelos por ola con prueba de Wald y descomposición Oaxaca-Blinder.
7. **Sensibilidad.**
8. **Figuras y exportación a Excel**, con el índice por persona.

**Definición.** Cada ítem vale 1 si la persona responde *sí* y 0 si responde *no*; Ns/Nc queda como faltante. El IPC es el porcentaje de los problemas respondidos que la persona reporta en su barrio, para quienes responden al menos 5 de los 8 ítems. Más alto indica peor convivencia.

$$\mathrm{IPC}_i = 100 \times \frac{\sum_{j \in R_i} \mathbb{1}[\text{sí}_{ij}]}{|R_i|}, \qquad |R_i| \geq 5$$

Todas las estimaciones usan `peso_az_estrato` normalizado a media 1, con errores robustos HC1. Equivale a `regress … [pw=…], robust` en Stata.

## 0. Instalación, librerías y carga del archivo

In [ ]:
!pip install -q girth openpyxl

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import os
import pandas as pd, numpy as np
import statsmodels.api as sm
from scipy import stats, optimize
import girth
import matplotlib
import matplotlib.pyplot as plt
pd.set_option('display.width', 220); pd.set_option('display.max_columns', 40)
SALIDA = 'salidas_ipc'; os.makedirs(SALIDA, exist_ok=True)

In [ ]:
# Opción A (por defecto): subir el Excel. Opción B: montar Drive y poner la ruta en RUTA.
try:
    from google.colab import files
    EN_COLAB = True
except ImportError:
    EN_COLAB = False
RUTA = 'EVPI_2024_2026_VF_1.xlsx'
# from google.colab import drive; drive.mount('/content/drive'); RUTA = '/content/drive/MyDrive/.../EVPI_2024_2026_VF_1.xlsx'
if EN_COLAB and not os.path.exists(RUTA):
    RUTA = list(files.upload())[0]
d = pd.read_excel(RUTA).copy()
print(d.shape)

## 1. Covariables

Las covariables se armonizan entre olas y se agrupan en cuatro bloques:

- **Sociodemográficas.**
- **Arraigo y cohesión barrial.**
- **Victimización y entorno:** victimización, confianza en la Policía y calles sin iluminación.
- **Percepción de inseguridad.**

Se excluyen el índice de problemas del barrio y la presencia de civiles armados, porque forman parte del fenómeno medido. Todos los modelos incluyen efectos fijos de administración zonal, con Eloy Alfaro como referencia.

In [ ]:
W, Y, AZ = 'peso_az_estrato', 'AÑO', 'ADMINISTRACION ZONAL'

def col(code):
    m = [c for c in d.columns if c.split(' - ')[0].strip() == code and '?/' not in c]
    assert len(m) == 1, code
    return d[m[0]].astype(str).str.strip()

X = pd.DataFrame(index=d.index)
X['w'] = d[W] / d[W].mean()
X['ola25'] = (d[Y] == 2025).astype(int)
X['az'] = d[AZ]
# ---------- resultado ----------
ch6 = col('CH_6')
X['ch6_cat'] = ch6
X['y'] = np.where(ch6 == '3. Disminuyó', 1, np.where(ch6.isin(['1. Aumentó', '2. Se mantuvo']), 0, np.nan))
X['y_na1'] = np.where(ch6 == '5. No aplica', 1, X['y'])       # cota superior: No aplica = disminuyó
X['y_na0'] = np.where(ch6 == '5. No aplica', 0, X['y'])       # cota inferior: No aplica = no disminuyó
X['y_ord'] = ch6.map({'1. Aumentó': 0, '2. Se mantuvo': 1, '3. Disminuyó': 2})
# ---------- covariables ----------
X['mujer'] = (col('DD_1') == 'Mujer').astype(int)
edad = d['RANGO ETAREO'].astype(str)
for k, v in {'e26_40': '26 A 40', 'e41_55': '41 A 55', 'e56_64': '56 A 64', 'e65': '65 Y MÁS'}.items():
    X[k] = (edad == v).astype(int)                                  # ref: 18-25
ed = col('DD_7').str[0]
X['ed_sec'] = ed.isin(['D', 'E']).astype(int)                       # ref: ninguno/primaria
X['ed_tec'] = ed.isin(['F', 'G']).astype(int)
X['ed_uni'] = ed.isin(['H', 'I', 'J', 'K', 'L', 'M']).astype(int)
X['nse_medio'] = (d['nse3'] == 2).astype(int)                       # ref: bajo
X['nse_alto'] = (d['nse3'] == 3).astype(int)
ing = col('DD_10').str[0]
X['ing_460_919'] = (ing == 'B').astype(int)                         # ref: < $459
X['ing_920'] = ing.isin(list('CDEFGHIJ')).astype(int)
X['ing_nsnc'] = (ing == 'K').astype(int)
X['trabaja'] = (col('DD_8') == '1. Sí').astype(int)
X['discap'] = (col('DD_3') == '1. Sí').astype(int)
X['extranj'] = (col('DD_4') == '2. No').astype(int)
X['unido'] = col('DD_5').str[0].isin(['A', 'D']).astype(int)
anios = pd.to_numeric(col('SP_1'), errors='coerce')
X['anios_barrio10'] = anios.clip(upper=60) / 10
X['vecinos_mal'] = col('SP_2').str[0].isin(['3', '4']).astype(int)   # no se lleva / no los conoce
X['poco_orgullo'] = col('SP_3').str[0].isin(['3', '4']).astype(int)
X['no_quiere_quedarse'] = (col('SP_4') == '2. No').astype(int)
X['participa'] = (col('SP_5') == '1. Sí').astype(int)
vic = pd.concat([(col(k) == '1. Sí').astype(int) for k in ['VI_7', 'VI_8', 'VI_9', 'VI_11', 'VI_12', 'VI_13']], axis=1)
X['vic_robo_viol'] = (col('VI_6') == '1. Sí').astype(int)
X['vic_otro_pers'] = vic.max(axis=1)
X['vic_vivienda'] = (col('VI_4') == '1. Sí').astype(int)
X['incivil'] = pd.concat([(col(f'CC_{i}') == '1. Sí').astype(int) for i in range(1, 10)], axis=1).sum(axis=1)
X['armas'] = (col('TA_3') == '1. Sí').astype(int)
cp = col('CIE_4_1')
X['conf_pol_no'] = (cp == '2. No').astype(int)                       # ref: confía
X['conf_pol_desc'] = (~cp.isin(['1. Sí', '2. No'])).astype(int)      # no conoce funciones
psc2 = col('PSC_2')
X['inseg_barrio'] = (psc2 == '1. Muy Inseguro + Inseguro').astype(int)
X['inseg_barrio_na'] = (psc2 == '98. No Aplica').astype(int)
X['delinc_aumento'] = (col('PSC_16') == '1. Aumentó').astype(int)
X['inseg_noche'] = (col('PSC_15') == '1. Muy Inseguro + Inseguro').astype(int)
X['inseg_dia'] = (col('PSC_14') == '1. Muy Inseguro + Inseguro').astype(int)
X['sec'] = d['sec']

AZS = sorted(X.az.unique()); REF_AZ = 'ELOY ALFARO'
for z in AZS:
    if z != REF_AZ: X['az_' + z.replace(' ', '_')] = (X.az == z).astype(int)
AZD = [c for c in X if c.startswith('az_')]

BLOQUES = {
 'Sociodemográficas': ['mujer','e26_40','e41_55','e56_64','e65','ed_sec','ed_tec','ed_uni','nse_medio','nse_alto',
                       'ing_460_919','ing_920','ing_nsnc','trabaja','discap','extranj','unido'],
 'Arraigo y cohesión barrial': ['anios_barrio10','vecinos_mal','poco_orgullo','no_quiere_quedarse','participa'],
 'Victimización y entorno': ['vic_robo_viol','vic_otro_pers','vic_vivienda','incivil','armas','conf_pol_no','conf_pol_desc'],
 'Percepción de inseguridad': ['inseg_barrio','inseg_barrio_na','delinc_aumento'],
}
LAB = {
 'ola25':'Ola 2025-2026 (vs. 2023-2024)','mujer':'Mujer','e26_40':'Edad 26-40 (vs. 18-25)','e41_55':'Edad 41-55','e56_64':'Edad 56-64',
 'e65':'Edad 65 y más','ed_sec':'Secundaria (vs. primaria o menos)','ed_tec':'Técnica/tecnológica','ed_uni':'Universitaria o más',
 'nse_medio':'NSE medio (vs. bajo)','nse_alto':'NSE alto','ing_460_919':'Ingreso hogar $460-919 (vs. <$459)','ing_920':'Ingreso hogar ≥ $920',
 'ing_nsnc':'Ingreso hogar NS/NC','trabaja':'Trabaja','discap':'Tiene discapacidad','extranj':'Nacionalidad extranjera',
 'unido':'Casado/a o unión libre','anios_barrio10':'Años en el barrio (por 10 años)','vecinos_mal':'No se lleva / no conoce a vecinos',
 'poco_orgullo':'Poco o nada orgulloso/a del barrio','no_quiere_quedarse':'No quiere seguir viviendo en el barrio',
 'participa':'Participa en iniciativa barrial de seguridad','vic_robo_viol':'Víctima de robo con violencia','vic_otro_pers':'Víctima de otro delito personal',
 'vic_vivienda':'Robo a la vivienda','incivil':'Problemas en el barrio (índice 0-9, por problema)','armas':'Vio civiles armados en el barrio',
 'conf_pol_no':'No confía en la Policía (vs. confía)','conf_pol_desc':'No conoce funciones de la Policía','inseg_barrio':'Se siente inseguro/a en su barrio',
 'inseg_barrio_na':'Inseguridad en barrio: no aplica','delinc_aumento':'Cree que la delincuencia aumentó en su barrio',
 'inseg_noche':'Inseguro/a caminando de noche','inseg_dia':'Inseguro/a caminando de día'}
for z in AZS:
    if z != REF_AZ: LAB['az_' + z.replace(' ', '_')] = f'AZ {z.title()} (vs. Eloy Alfaro)'

GROUPS = {'edad': ['e26_40','e41_55','e56_64','e65'], 'educ': ['ed_sec','ed_tec','ed_uni'], 'nse': ['nse_medio','nse_alto'],
          'ing': ['ing_460_919','ing_920','ing_nsnc'], 'conf': ['conf_pol_no','conf_pol_desc'], 'insb': ['inseg_barrio','inseg_barrio_na'], 'az': AZD}
CONT = ['anios_barrio10', 'incivil']

def spec(k):
    v = ['ola25']
    for b in list(BLOQUES)[:k]: v += BLOQUES[b]
    return v + AZD

# ---------- estimación ----------

In [ ]:
_B0 = BLOQUES  # bloques generales; el IPC excluye problemas del barrio y civiles armados
BLOQUES = {k: [v for v in vs if v not in ('incivil', 'armas')] for k, vs in _B0.items()}
BLOQUES['Victimización y entorno'] = BLOQUES['Victimización y entorno'] + ['sin_ilum']
LAB['sin_ilum'] = 'Calles sin iluminación en el barrio'
def spec(k):
    v = ['ola25']
    for b in list(BLOQUES)[:k]: v += BLOQUES[b]
    return v + AZD

# Calles sin iluminación (CC_7) como variable explicativa: 1 = sí, 0 = no o Ns/Nc
X['sin_ilum'] = (col('CC_7') == '1. Sí').astype(int)
print(BLOQUES['Victimización y entorno'])

## 2. Ítems de la sección de convivencia

% que reporta cada problema (entre quienes responden sí o no) y % de Ns/Nc sobre el total, por ola. Calles sin iluminación (CC_7) no entra en el índice.

In [ ]:
# Funciones psicométricas: tetracóricas, un factor (ejes principales), α ordinal, KR-20 e IRT 2PL
def tetra(x, y, ww=None):
    m = ~(np.isnan(x) | np.isnan(y)); x, y = x[m], y[m]; ww = np.ones(len(x)) if ww is None else ww[m]
    n11 = ww[(x == 1) & (y == 1)].sum(); n10 = ww[(x == 1) & (y == 0)].sum(); n01 = ww[(x == 0) & (y == 1)].sum(); n00 = ww[(x == 0) & (y == 0)].sum()
    N = n11 + n10 + n01 + n00
    tx = stats.norm.ppf((n00 + n01) / N); ty = stats.norm.ppf((n00 + n10) / N)
    def nll(r):
        p00 = stats.multivariate_normal.cdf([tx, ty], cov=[[1, r], [r, 1]])
        px0 = stats.norm.cdf(tx); py0 = stats.norm.cdf(ty)
        p01 = px0 - p00; p10 = py0 - p00; p11 = 1 - px0 - py0 + p00
        ps = np.clip([p00, p01, p10, p11], 1e-12, 1)
        return -(n00*np.log(ps[0]) + n01*np.log(ps[1]) + n10*np.log(ps[2]) + n11*np.log(ps[3]))
    return optimize.minimize_scalar(nll, bounds=(-.99, .99), method='bounded').x
def tetra_mat(Bm, ww):
    k = Bm.shape[1]; M = np.eye(k)
    for i in range(k):
        for j in range(i + 1, k):
            M[i, j] = M[j, i] = tetra(Bm[:, i], Bm[:, j], ww)
    return M
def one_factor(M):
    # factorización de ejes principales iterada (1 factor)
    h = 1 - 1 / np.diag(np.linalg.inv(M))
    for _ in range(500):
        Mr = M.copy(); np.fill_diagonal(Mr, h)
        ev, vec = np.linalg.eigh(Mr); l = vec[:, -1] * np.sqrt(max(ev[-1], 0))
        hn = l**2
        if np.max(np.abs(hn - h)) < 1e-7: break
        h = hn
    l = l if l.sum() > 0 else -l
    return l, np.linalg.eigvalsh(M)[::-1]
def alpha_from_corr(M):
    k = M.shape[0]; return k / (k - 1) * (1 - k / M.sum())
def kr20(df):
    k = df.shape[1]; p = df.mean(); return k / (k - 1) * (1 - (p*(1-p)).sum() / df.sum(axis=1).var(ddof=0))
def irt_fit(Bm):
    data = np.where(np.isnan(Bm), girth.INVALID_RESPONSE, Bm).astype(int).T
    est = girth.twopl_mml(data)
    return est, data

In [ ]:
ITEMS = ['CC_1', 'CC_2', 'CC_3', 'CC_4', 'CC_5', 'CC_6', 'CC_8', 'CC_9']   # CC_7 (iluminación) sale del índice y pasa a ser explicativa
ILAB = {'CC_1': 'Peleas o riñas en el espacio público', 'CC_2': 'Pandillas o grupos delictivos en el espacio público',
        'CC_3': 'Consumo de alcohol en el espacio público', 'CC_4': 'Venta clandestina de alcohol',
        'CC_5': 'Consumo de drogas en el espacio público', 'CC_6': 'Venta de drogas en el espacio público',
        'CC_8': 'Apropiación ilegal de viviendas por grupos delictivos',
        'CC_9': 'Reclutamiento de niñas, niños y adolescentes'}
B = pd.DataFrame(index=d.index); NS = pd.DataFrame(index=d.index)
for k in ITEMS + ['CC_7']:
    s = col(k)
    B[k] = np.where(s == '1. Sí', 1.0, np.where(s == '2. No', 0.0, np.nan))
    NS[k] = (~s.isin(['1. Sí', '2. No'])).astype(int)
w, ola = X.w, X.ola25
P = {'ILAB': ILAB}
rows = []
for k in ITEMS:
    for o in [0, 1]:
        m = ola == o; mv = m & B[k].notna()
        rows.append(dict(item=k, ola=o, pct_si=np.average(B.loc[mv, k], weights=w[mv]), pct_ns=np.average(NS.loc[m, k], weights=w[m]), n=int(mv.sum())))
P['items'] = pd.DataFrame(rows)

P['items'].round(3)

## 3. Validación psicométrica

- Correlaciones **tetracóricas** ponderadas y **un factor** por ejes principales iterados, con autovalores, cargas y α ordinal.
- **KR-20** en casos completos, correlación ítem-resto y α si se elimina cada ítem.
- **Congruencia de Tucker** entre las cargas de ambas olas.

In [ ]:
psy = {}
for name, msk in [('Conjunto', np.ones(len(X), bool)), ('2023-2024', (ola == 0).values), ('2025-2026', (ola == 1).values)]:
    M = tetra_mat(B[ITEMS].values[msk], w.values[msk]); lo, ev = one_factor(M)
    psy[name] = dict(tetra=pd.DataFrame(M, ITEMS, ITEMS), loadings=pd.Series(lo, ITEMS), eig=ev)
P['psy'] = psy
l23, l25 = psy['2023-2024']['loadings'], psy['2025-2026']['loadings']
P['tucker'] = float((l23 @ l25) / np.sqrt((l23 @ l23) * (l25 @ l25)))
P['alpha_ord'] = {k: alpha_from_corr(v['tetra'].values) for k, v in psy.items()}
cc = B[ITEMS].dropna()
P['kr20'] = {'Conjunto': kr20(cc), '2023-2024': kr20(cc[ola[cc.index] == 0]), '2025-2026': kr20(cc[ola[cc.index] == 1]), 'n': len(cc)}
P['item_rest'] = {k: np.corrcoef(cc[k], cc.drop(columns=k).sum(axis=1))[0, 1] for k in ITEMS}
P['alpha_if_deleted'] = {k: alpha_from_corr(psy['Conjunto']['tetra'].drop(index=k, columns=k).values) for k in ITEMS}
# 2 factores (exploratorio): autovalores ya dan la pista; cargas sin rotar por si hay estructura secundaria

print('Autovalores:', {k: np.round(v['eig'], 2) for k, v in P['psy'].items()})
print('α ordinal:', {k: round(v, 3) for k, v in P['alpha_ord'].items()}, '| KR-20:', {k: round(v, 3) for k, v in P['kr20'].items()})
print('Tucker:', round(P['tucker'], 3))
pd.DataFrame({k: v['loadings'] for k, v in P['psy'].items()}).assign(alpha_sin_item=pd.Series(P['alpha_if_deleted'])).round(3)

### IRT 2PL y funcionamiento diferencial (DIF) por ola

El modelo IRT se estima con `girth`, por máxima verosimilitud marginal y sin pesos. El DIF se prueba con un logit de cada ítem sobre el puntaje en los demás ítems y la ola (DIF uniforme), y luego sobre su interacción (DIF no uniforme).

In [ ]:
est, data = irt_fit(B[ITEMS].values)
P['irt'] = pd.DataFrame({'a': est['Discrimination'], 'b': est['Difficulty']}, index=ITEMS)
P['irt_wave'] = {o: pd.DataFrame(dict(zip(['a', 'b'], [irt_fit(B[ITEMS].values[(ola == o).values])[0][k] for k in ['Discrimination', 'Difficulty']])), index=ITEMS) for o in [0, 1]}
theta = girth.ability_eap(data, est['Difficulty'], est['Discrimination'])
dif = []
for k in ITEMS:
    rest = [j for j in ITEMS if j != k]
    m = B[k].notna() & (B[rest].notna().sum(axis=1) >= 4)
    dd = pd.DataFrame({'y': B.loc[m, k], 'rest': B.loc[m, rest].mean(axis=1), 'ola': ola[m]}); dd['int'] = dd.rest * dd.ola
    r1 = sm.GLM(dd.y, sm.add_constant(dd[['rest', 'ola']]), family=sm.families.Binomial(), var_weights=w[m]).fit(cov_type='HC1')
    r2 = sm.GLM(dd.y, sm.add_constant(dd[['rest', 'ola', 'int']]), family=sm.families.Binomial(), var_weights=w[m]).fit(cov_type='HC1')
    dif.append(dict(item=k, or_ola=np.exp(r1.params['ola']), p_unif=r1.pvalues['ola'], p_nounif=r2.pvalues['int'], n=int(m.sum())))
P['dif'] = pd.DataFrame(dif)

display(P['irt'].round(3))
P['dif'].round(3)

## 4. Construcción del índice, subíndices y versiones

| Variable | Definición |
|---|---|
| `ipc` | Principal: 8 ítems, mínimo 5 respondidos, 0-100 |
| `desorden` | Subíndice de desorden cotidiano (CC_1 a CC_6), mínimo 4 |
| `criminal` | Presencia criminal grave: apropiación de viviendas o reclutamiento (0/1) |
| `ipc_9` | Con calles sin iluminación (9 ítems), para comparar |
| `ipc_ns0` / `ipc_ns1` | Ns/Nc = no / Ns/Nc = sí |
| `ipc_irt` | Puntaje EAP del 2PL, en desviaciones estándar |

In [ ]:
# ---------- índices (orientación: 0 = ningún problema, 100 = todos) ----------
MINI = 5
nval = B[ITEMS].notna().sum(axis=1)
IDX = pd.DataFrame(index=d.index); IDX['n_items'] = nval
def idx(items, mn): v = B[items].notna().sum(axis=1); return np.where(v >= mn, B[items].mean(axis=1) * 100, np.nan)
IDX['ipc'] = idx(ITEMS, MINI)
IDX['ipc_9'] = idx(ITEMS + ['CC_7'], 6)                                  # con iluminación (versión original)
IDX['ipc_ns0'] = B[ITEMS].fillna(0).mean(axis=1) * 100                 # Ns/Nc = no
IDX['ipc_ns1'] = B[ITEMS].fillna(1).mean(axis=1) * 100                 # Ns/Nc = sí
IDX['ipc_irt'] = np.where(nval >= MINI, theta, np.nan)
IDX['desorden'] = idx(['CC_1', 'CC_2', 'CC_3', 'CC_4', 'CC_5', 'CC_6'], 4)
IDX['criminal'] = np.where(B[['CC_8', 'CC_9']].notna().sum(axis=1) >= 1, B[['CC_8', 'CC_9']].max(axis=1), np.nan)
IDX['conteo'] = np.where(nval >= MINI, B[ITEMS].sum(axis=1), np.nan)
IDX['alguno'] = np.where(nval >= MINI, (B[ITEMS].max(axis=1)).astype(float), np.nan)
IDX['grave'] = np.where(B[['CC_2', 'CC_6', 'CC_8', 'CC_9']].notna().sum(axis=1) >= 3, B[['CC_2', 'CC_6', 'CC_8', 'CC_9']].max(axis=1), np.nan)
IDX['sin_ilum'] = (B['CC_7'] == 1).astype(int)                          # explicativa (Ns/Nc = 0)
P['corr_idx'] = IDX[['ipc', 'ipc_9', 'ipc_ns0', 'ipc_ns1', 'ipc_irt', 'desorden']].corr()
P['cover'] = {o: dict(n=int((ola == o).sum()), con_indice=int(IDX.loc[ola == o, 'ipc'].notna().sum()), media_items=float(nval[ola == o].mean())) for o in [0, 1]}
IDX['ola25'] = ola; IDX['w'] = w
print(P['cover'])
P['corr_idx'].round(3)

## 5. Cambio entre olas: medias, distribución y subgrupos

In [ ]:
def _SG(X):
    return {
 'Sexo': [('Hombre', X.mujer == 0), ('Mujer', X.mujer == 1)],
 'Edad': [('18-25', (X[['e26_40','e41_55','e56_64','e65']].sum(axis=1) == 0)), ('26-40', X.e26_40 == 1), ('41-55', X.e41_55 == 1), ('56-64', X.e56_64 == 1), ('65 y más', X.e65 == 1)],
 'Nivel socioeconómico': [('Bajo', (X.nse_medio + X.nse_alto) == 0), ('Medio', X.nse_medio == 1), ('Alto', X.nse_alto == 1)],
 'Educación': [('Primaria o menos', (X.ed_sec + X.ed_tec + X.ed_uni) == 0), ('Secundaria', X.ed_sec == 1), ('Técnica/tecnológica', X.ed_tec == 1), ('Universitaria o más', X.ed_uni == 1)],
 'Victimización personal (12 meses)': [('No víctima', (X.vic_robo_viol + X.vic_otro_pers) == 0), ('Víctima', (X.vic_robo_viol + X.vic_otro_pers) > 0)],
 'Problemas en el barrio': [('0-2 problemas', X.incivil <= 2), ('3-5 problemas', X.incivil.between(3, 5)), ('6-9 problemas', X.incivil >= 6)],
 'Percepción en el barrio': [('Se siente seguro/a', X.inseg_barrio == 0), ('Se siente inseguro/a', X.inseg_barrio == 1)],
 'Administración zonal': [(z.title().replace('Saenz', 'Sáenz').replace('Calderon', 'Calderón'), X.az == z) for z in AZS],
}

for c in ['ipc', 'ipc_9', 'ipc_ns0', 'ipc_ns1', 'ipc_irt', 'desorden', 'criminal', 'conteo', 'alguno', 'grave', 'n_items', 'sin_ilum']: X[c] = IDX[c]
R = {}

def wls(df, yv, xs, wv='w', cluster=None):
    m = df[[yv, wv] + xs].notna().all(axis=1); dd = df[m]
    mod = sm.WLS(dd[yv], sm.add_constant(dd[xs].astype(float)), weights=dd[wv])
    r = mod.fit(cov_type='HC1') if cluster is None else mod.fit(cov_type='cluster', cov_kwds={'groups': pd.factorize(dd[cluster])[0]})
    return r, dd

def change(df, yv, wv='w', cluster=None):
    r, dd = wls(df, yv, ['ola25'], wv, cluster)
    g0, g1 = dd[dd.ola25 == 0], dd[dd.ola25 == 1]
    return dict(m23=np.average(g0[yv], weights=g0[wv]), m25=np.average(g1[yv], weights=g1[wv]),
                dif=r.params['ola25'], se=r.bse['ola25'], p=r.pvalues['ola25'], n=int(r.nobs))

# 1. Cambio por versión del índice
VERS = {'ipc': 'Principal: % de los 8 problemas presentes en el barrio (mín. 5 respondidos)', 'desorden': 'Subíndice de desorden cotidiano (CC_1 a CC_6)',
        'criminal': 'Presencia criminal grave: apropiación de viviendas o reclutamiento de NNA (0/1)', 'ipc_9': 'Con calles sin iluminación (9 ítems, versión original)',
        'ipc_ns0': 'Ns/Nc = no (8 ítems, todos los casos)', 'ipc_ns1': 'Ns/Nc = sí (8 ítems, todos los casos)',
        'ipc_irt': 'Puntaje IRT 2PL (EAP, desviaciones estándar)', 'conteo': 'Número de problemas reportados (0-8)',
        'alguno': 'Reporta al menos un problema (0/1)', 'grave': 'Reporta pandillas, venta de drogas, apropiación de viviendas o reclutamiento (0/1)'}
R['change'] = pd.DataFrame([dict(version=k, desc=v, **change(X, k)) for k, v in VERS.items()])
X['w1'] = 1.0
extra = [dict(version='ipc', desc='Principal, sin ponderar', **change(X, 'ipc', 'w1')),
         dict(version='ipc', desc='Principal, EE agrupados por sector censal', **change(X, 'ipc', cluster='sec'))]
R['change'] = pd.concat([R['change'], pd.DataFrame(extra)], ignore_index=True)

# distribución del índice por ola (bins)
bins = [-.1, 0.001, 25, 50, 75, 99.999, 100]; labs = ['0 (ninguno)', '1-25', '26-50', '51-75', '76-99', '100 (todos)']
dist = []
for o in [0, 1]:
    g = X[(X.ola25 == o) & X.ipc.notna()]
    cat = pd.cut(g.ipc, bins=bins, labels=labs)
    for l in labs: dist.append(dict(ola=o, tramo=l, pct=np.average(cat == l, weights=g.w)))
R['dist'] = pd.DataFrame(dist)

# 2. Subgrupos
def SUBG_DEF(X):
    s = _SG(X); s.pop('Problemas en el barrio')
    s['Calles sin iluminación'] = [('Sin problema de iluminación', X.sin_ilum == 0), ('Con calles sin iluminación', X.sin_ilum == 1)]
    s['Inseguridad caminando de noche'] = [('Segura/o o no aplica', X.inseg_noche == 0), ('Insegura/o', X.inseg_noche == 1)]
    return s
rows = []
for grp, cats in SUBG_DEF(X).items():
    for lab, m in cats:
        c = change(X[m], 'ipc'); rows.append(dict(grupo=grp, categoria=lab, **c))
R['subg'] = pd.DataFrame(rows)


R['change'].round(3)

In [ ]:
R['subg'].round(2)

## 6. Determinantes del índice

### 6.1 Modelos anidados y por ola

MCO ponderado con errores HC1. Se ajustan cuatro modelos que van sumando bloques de covariables:

1. **M1:** sociodemográficas.
2. **M2:** M1 + arraigo y cohesión barrial.
3. **M3:** M2 + victimización, confianza en la Policía y calles sin iluminación.
4. **M4:** M3 + percepción de inseguridad.

Todos incluyen la ola y efectos fijos de administración zonal. Luego se estima M4 por ola y una prueba de Wald conjunta de las interacciones ola × covariable.

In [ ]:
# 3. Determinantes (MCO ponderado sobre índice 0-100)
models = {}
for k, name in enumerate(['M1 Sociodemográficas', 'M2 + Arraigo y cohesión', 'M3 + Victimización y entorno', 'M4 + Percepción'], 1):
    r, dd = wls(X, 'ipc', spec(k))
    models[name] = dict(params=r.params, bse=r.bse, p=r.pvalues, n=int(r.nobs), r2=r.rsquared)
R['models'] = models
xs4 = spec(4); xs4w = [v for v in xs4 if v != 'ola25']
by = {}
for o in [0, 1]:
    r, dd = wls(X[X.ola25 == o], 'ipc', xs4w); by[o] = pd.DataFrame({'b': r.params, 'se': r.bse, 'p': r.pvalues})
cmp_ = by[0].join(by[1], lsuffix='_23', rsuffix='_25').drop('const')
cmp_['dif'] = cmp_.b_25 - cmp_.b_23; cmp_['se_dif'] = np.sqrt(cmp_.se_23**2 + cmp_.se_25**2)
cmp_['p_dif'] = 2 * (1 - stats.norm.cdf(np.abs(cmp_.dif / cmp_.se_dif)))
R['by_wave'] = cmp_
Xi = X.copy(); inter = []
for v in xs4w: Xi['x_' + v] = Xi[v] * Xi.ola25; inter.append('x_' + v)
r, dd = wls(Xi, 'ipc', ['ola25'] + xs4w + inter)
for nm, sel in [('wald', inter), ('wald_noaz', ['x_' + v for v in xs4w if v not in AZD])]:
    t = r.wald_test(' = 0, '.join(sel) + ' = 0', scalar=True)
    R[nm] = dict(chi2=float(t.statistic), df=len(sel), p=float(t.pvalue))


for k, v in R['models'].items(): print(f"{k}: n = {v['n']}, R² = {v['r2']:.3f}, ola = {v['params']['ola25']:.2f}")
print('Wald (sin AZ):', R['wald_noaz'])
M4 = R['models']['M4 + Percepción']
pd.DataFrame({'b': M4['params'], 'EE': M4['bse'], 'p': M4['p']}).rename(index=LAB).round(3)

In [ ]:
R['by_wave'][['b_23', 'p_23', 'b_25', 'p_25', 'dif', 'p_dif']].rename(index=LAB).round(3)

### 6.2 Descomposición Oaxaca-Blinder

La descomposición tiene dos partes, explicada y no explicada. Los intervalos se obtienen por bootstrap estratificado por ola, con 300 réplicas. Tarda alrededor de 1 minuto.

In [ ]:
# 4. Oaxaca-Blinder
def oaxaca(df, xs, blocks, yv='ipc'):
    df = df[df[yv].notna()]; g0, g1 = df[df.ola25 == 0], df[df.ola25 == 1]
    m0 = np.average(g0[xs], axis=0, weights=g0.w); m1 = np.average(g1[xs], axis=0, weights=g1.w)
    bp = sm.WLS(df[yv], sm.add_constant(df[['ola25'] + xs].astype(float)), weights=df.w).fit().params[xs].values
    y0 = np.average(g0[yv], weights=g0.w); y1 = np.average(g1[yv], weights=g1.w)
    c = (m1 - m0) * bp
    res = {'total': y1 - y0, 'explicado': c.sum(), 'no_explicado': (y1 - y0) - c.sum()}
    for b, vs in blocks.items(): res['E: ' + b] = sum(c[xs.index(v)] for v in vs if v in xs)
    return res
blocks = {**BLOQUES, 'Administración zonal': AZD}
pt = oaxaca(X, xs4w, blocks)
rng = np.random.default_rng(2026); base = X[X.ipc.notna()]
bt = pd.DataFrame([oaxaca(pd.concat([g.sample(len(g), replace=True, random_state=int(rng.integers(1e9))) for _, g in base.groupby('ola25')]), xs4w, blocks) for _ in range(300)])
R['oaxaca'] = pd.DataFrame({'componente': list(pt), 'estimado': list(pt.values()), 'se': [bt[k].std() for k in pt],
                            'lo': [bt[k].quantile(.025) for k in pt], 'hi': [bt[k].quantile(.975) for k in pt]})
pt2 = oaxaca(X, xs4w, blocks, 'desorden')
R['oaxaca_alt'] = pt2


print('Subíndice de desorden:', {k: round(v, 2) for k, v in R['oaxaca_alt'].items()})
R['oaxaca'].round(2)

## 7. Sensibilidad del efecto de la ola (M4)

In [ ]:
# 5. Sensibilidad del efecto de ola condicional
sens = []
def addS(desc, yv='ipc', xs=None, df=None, wv='w', cluster=None):
    r, dd = wls(X if df is None else df, yv, xs4 if xs is None else xs, wv=wv, cluster=cluster)
    sens.append(dict(especificacion=desc, ola=r.params['ola25'], se=r.bse['ola25'], p=r.pvalues['ola25'], n=int(r.nobs)))
addS('Principal, M4')
for yv, desc in [('desorden', 'Subíndice de desorden cotidiano, M4'), ('ipc_9', 'Con iluminación (9 ítems), M4 sin iluminación como explicativa'), ('ipc_ns0', 'Ns/Nc = no, M4'), ('ipc_ns1', 'Ns/Nc = sí, M4'), ('ipc_irt', 'IRT (DE), M4')]:
    addS(desc, yv, xs=[v for v in xs4 if v != 'sin_ilum'] if yv == 'ipc_9' else None)
addS('Principal, M4, EE agrupados por sector', cluster='sec'); addS('Principal, M4, sin ponderar', wv='w1')
addS('Principal, M3 (sin percepción)', xs=spec(3))
X['todos8'] = X.n_items == 8; addS('Principal, M4, solo quienes responden los 8 ítems', df=X[X.todos8])
addS('Principal, M4 sin iluminación como explicativa', xs=[v for v in xs4 if v != 'sin_ilum'])
R['sens'] = pd.DataFrame(sens)

R['sens'].round(3)

## 8. Figuras

In [ ]:
NAVY, BLUE, CYAN, GREEN, ORANGE, AMBER, GREY = '#15206b', '#4e59aa', '#1d9ebe', '#0fa27c', '#ff6840', '#ffb500', '#94a3b8'
plt.rcParams.update({'text.parse_math': False, 'font.family': 'DejaVu Sans', 'font.size': 9, 'axes.spines.top': False, 'axes.spines.right': False})
def es(v, d=1): return f'{v:.{d}f}'.replace('.', ',').replace('-', '−')
SRC = 'Fuente: EVPI 2023-2024 y 2025-2026, OMSCGR. Ponderado con peso_az_estrato.'
ILAB = {'CC_1': 'Peleas\no riñas', 'CC_2': 'Pandillas o grupos\ndelictivos', 'CC_3': 'Consumo de\nalcohol', 'CC_4': 'Venta clandestina\nde alcohol',
        'CC_5': 'Consumo de\ndrogas', 'CC_6': 'Venta de\ndrogas', 'CC_8': 'Apropiación\nde viviendas', 'CC_9': 'Reclutamiento\nde NNA'}
ITEMS = list(ILAB)

# Fig A: ítems y psicometría
fig, axs = plt.subplots(1, 2, figsize=(15, 5.2), gridspec_kw={'width_ratios': [1.7, 1]})
ax = axs[0]; I = P['items']; x = np.arange(len(ITEMS))
for o, c, dx in [(0, BLUE, -.2), (1, ORANGE, .2)]:
    v = I[I.ola == o].set_index('item').loc[ITEMS]
    ax.bar(x + dx, v.pct_si*100, width=.38, color=c, label='2023-2024' if o == 0 else '2025-2026')
    for i, (val, na) in enumerate(zip(v.pct_si, v.pct_ns)):
        ax.text(i + dx, val*100 + 1, es(val*100, 0), ha='center', fontsize=7.5, color=NAVY)
        ax.text(i + dx, 1.5, f'NS\n{es(na*100, 0)}%', ha='center', fontsize=6, color='white')
ax.set_xticks(x); ax.set_xticklabels([ILAB[k] for k in ITEMS], fontsize=6.8); ax.set_ylim(0, 90)
ax.set_ylabel('% que reporta el problema en su barrio'); ax.legend(frameon=False, fontsize=8, loc='upper right', ncol=2)
ax.set_title('Problemas de convivencia reportados en el barrio, por ola\n(NS = % Ns/Nc sobre el total; excluido del porcentaje)', loc='left', fontsize=10.5, color=NAVY, fontweight='bold')
ax = axs[1]
L = pd.DataFrame({k: P['psy'][k]['loadings'] for k in ['2023-2024', '2025-2026']}).loc[ITEMS]
y = np.arange(len(ITEMS))
ax.barh(y - .2, L['2023-2024'], height=.38, color=BLUE, label='2023-2024'); ax.barh(y + .2, L['2025-2026'], height=.38, color=ORANGE, label='2025-2026')
ax.set_yticks(y); ax.set_yticklabels([ILAB[k].replace('\n', ' ') for k in ITEMS], fontsize=8); ax.invert_yaxis(); ax.set_xlim(0, 1)
ax.set_xlabel('Carga factorial (1 factor, correlaciones tetracóricas)')
ev = P['psy']['Conjunto']['eig']
ax.set_title(f"Un factor dominante, estable entre olas\nAutovalores: {es(ev[0], 2)} / {es(ev[1], 2)} · α ordinal = {es(P['alpha_ord']['Conjunto'], 2)}\nCongruencia de Tucker = {es(P['tucker'], 3)}",
             loc='left', fontsize=10.5, color=NAVY, fontweight='bold')
ax.legend(frameon=False, fontsize=8, loc='lower right')
fig.text(.01, .01, SRC, fontsize=7.5, color='#64748b'); plt.tight_layout(rect=(0, .04, 1, 1))
plt.savefig(f'{SALIDA}/ipc_fig1_items.png', dpi=170); plt.show()

# Fig B: distribución + cambio por subgrupos clave y AZ
fig, axs = plt.subplots(1, 2, figsize=(12, 5.2), gridspec_kw={'width_ratios': [1, 1.2]})
ax = axs[0]; D = R['dist']; labs = list(dict.fromkeys(D.tramo)); x = np.arange(len(labs))
for o, c, dx in [(0, BLUE, -.2), (1, ORANGE, .2)]:
    v = D[D.ola == o].set_index('tramo').loc[labs].pct*100
    ax.bar(x + dx, v, width=.38, color=c, label='2023-2024' if o == 0 else '2025-2026')
    for i, val in enumerate(v): ax.text(i + dx, val + .6, es(val, 0), ha='center', fontsize=7.5, color=NAVY)
ch = R['change'].iloc[0]
ax.set_xticks(x); ax.set_xticklabels(labs, fontsize=8); ax.set_xlabel('Índice de problemas de convivencia (0-100)'); ax.set_ylabel('% de personas')
ax.legend(frameon=False, fontsize=8)
ax.set_title(f"Distribución del índice de problemas de convivencia\nMedia: {es(ch.m23)} → {es(ch.m25)} (+{es(ch.dif)} puntos, p < 0,001)", loc='left', fontsize=10.5, color=NAVY, fontweight='bold')
ax = axs[1]; S = R['subg']
keep = [('Victimización personal (12 meses)', None), ('Calles sin iluminación', None), ('Administración zonal', None)]
rows = []
for g, _ in keep:
    rows.append(('__' + g, None))
    sub = S[S.grupo == g]
    if g == 'Administración zonal': sub = sub.sort_values('dif')
    rows += [(r.categoria, r) for r in sub.itertuples()]
yy = 0; ticks, tl = [], []
for lab, r in rows:
    if r is None:
        ax.text(-14, yy, lab[2:], fontsize=8.5, fontweight='bold', color=NAVY, va='center'); yy += 1; continue
    c = (ORANGE if r.dif > 0 else GREEN) if r.p < .05 else GREY
    ax.plot([r.dif - 1.96*r.se, r.dif + 1.96*r.se], [yy, yy], color=c, lw=1.6)
    ax.scatter(r.dif, yy, color=c, s=30, zorder=3); ticks.append(yy); tl.append(lab); yy += 1
ax.axvline(0, color='#555', lw=.8, ls='--'); ax.axvline(ch.dif, color=NAVY, lw=.8, ls=':')
ax.text(ch.dif, -1.3, 'Media DMQ', ha='center', fontsize=7.5, color=NAVY)
ax.set_yticks(ticks); ax.set_yticklabels(tl, fontsize=8); ax.set_ylim(yy - .5, -2); ax.set_xlim(-14, 24)
ax.set_xlabel('Cambio en el índice 2023-2024 → 2025-2026 (puntos), IC 95%')
ax.set_title('Cambio del índice por grupo (puntos)\nmás deterioro entre víctimas y en barrios iluminados', loc='left', fontsize=10.5, color=NAVY, fontweight='bold')
fig.text(.01, .01, SRC + ' Naranja: más problemas; verde: menos; gris: p ≥ 0,05.', fontsize=7.5, color='#64748b'); plt.tight_layout(rect=(0, .04, 1, 1))
plt.savefig(f'{SALIDA}/ipc_fig2_cambio.png', dpi=170); plt.show()

# Fig C: determinantes (conjunto y por ola)
M4 = R['models']['M4 + Percepción']; Bw = R['by_wave']
sel = [v for v in sum(BLOQUES.values(), []) if v != 'inseg_barrio_na']
fig, axs = plt.subplots(1, 2, figsize=(13, 10.5), sharey=True)
for ax, mode in zip(axs, ['pool', 'wave']):
    for i, v in enumerate(sel):
        if mode == 'pool':
            b, se, p = M4['params'][v], M4['bse'][v], M4['p'][v]
            c = (ORANGE if b > 0 else GREEN) if p < .05 else GREY
            ax.plot([b - 1.96*se, b + 1.96*se], [i, i], color=c, lw=1.6); ax.scatter(b, i, color=c, s=30 if p < .05 else 16, zorder=3)
        else:
            for o, c, dy in [('23', BLUE, -.17), ('25', ORANGE, .17)]:
                b, se, p = Bw.loc[v, 'b_' + o], Bw.loc[v, 'se_' + o], Bw.loc[v, 'p_' + o]
                ax.plot([b - 1.96*se, b + 1.96*se], [i + dy]*2, color=c, lw=1.3, alpha=1 if p < .05 else .4)
                ax.scatter(b, i + dy, color=c, s=24, alpha=1 if p < .05 else .4, zorder=3, label=('2023-2024' if o == '23' else '2025-2026') if i == 0 else None)
            if Bw.loc[v, 'p_dif'] < .05: ax.text(22.5, i, '◆', color=NAVY, fontsize=9, va='center', ha='center')
    ax.axvline(0, color='#555', lw=.8, ls='--'); ax.set_xlim(-14, 24)
    ax.set_xlabel('Efecto sobre el índice (puntos de 0-100), IC 95%')
axs[0].set_yticks(range(len(sel))); axs[0].set_yticklabels([LAB[v] for v in sel], fontsize=8); axs[0].invert_yaxis()
axs[0].set_title(f"Determinantes del índice (ambas olas)\nMCO ponderado con EF de ola y AZ · R² = {es(M4['r2'], 3)}", loc='left', fontsize=10.5, color=NAVY, fontweight='bold')
W = R['wald_noaz']
axs[1].set_title(f"Por ola · ◆ diferencia significativa (p < 0,05)\nPrueba conjunta: χ²({W['df']}) = {es(W['chi2'])}, p < 0,001", loc='left', fontsize=10.5, color=NAVY, fontweight='bold')
axs[1].legend(frameon=False, fontsize=8.5, loc='lower right')
fig.text(.01, .005, SRC + ' Naranja: más problemas de convivencia; verde: menos; gris/tenue: p ≥ 0,05.', fontsize=7.5, color='#64748b')
plt.tight_layout(rect=(0, .015, 1, 1)); plt.savefig(f'{SALIDA}/ipc_fig3_determinantes.png', dpi=170); plt.show()

## 9. Exportación a Excel

El archivo replica el Excel del informe. Incluye el **índice por persona**, con `ID_Todo`, para unirlo a la base. Trae el IPC, los subíndices, las versiones y la variable de iluminación.

In [ ]:
import pickle, pandas as pd, numpy as np
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment
from openpyxl.drawing.image import Image as XLImage
from openpyxl.utils import get_column_letter as L
from openpyxl.formatting.rule import FormulaRule
F = 'Arial'; fnt = lambda **k: Font(name=F, **k)
HDR = PatternFill('solid', fgColor='1F4E79'); BLK = PatternFill('solid', fgColor='D9E2F3'); BIN = '0000FF'
ITEMS = ['CC_1', 'CC_2', 'CC_3', 'CC_4', 'CC_5', 'CC_6', 'CC_8', 'CC_9']
ILAB = P['ILAB']
def head(ws, r, hs, h=30):
    for j, t in enumerate(hs, 1):
        c = ws.cell(r, j, t); c.font = fnt(bold=True, color='FFFFFF', size=9); c.fill = HDR
        c.alignment = Alignment(wrap_text=True, horizontal='center', vertical='center')
    ws.row_dimensions[r].height = h
def title(ws, t, sub=None):
    ws['A1'] = t; ws['A1'].font = fnt(bold=True, size=13)
    if sub: ws['A2'] = sub; ws['A2'].font = fnt(italic=True, size=9, color='555555')
def widths(ws, w):
    for j, x in enumerate(w, 1): ws.column_dimensions[L(j)].width = x
def star(p): return '***' if p < .001 else '**' if p < .01 else '*' if p < .05 else '†' if p < .10 else ''
def put(ws, r, vals, fmts={}, blue=()):
    for j, v in enumerate(vals, 1):
        c = ws.cell(r, j, v); c.font = fnt(size=9, color=BIN if j in blue else '000000')
        if j in fmts: c.number_format = fmts[j]
def es(v, d=1): return f'{v:.{d}f}'.replace('.', ',').replace('-', '−')

Sg = lambda g, c: S.loc[(g, c)]
ch = R['change'].drop_duplicates('version').set_index('version'); ob = R['oaxaca'].set_index('componente'); M4 = R['models']['M4 + Percepción']; Bw = R['by_wave']
S = R['subg'].set_index(['grupo', 'categoria'])
wb = Workbook()
ws = wb.active; ws.title = 'Notas'
I = P['items']
c0 = R['change'].iloc[0]
it = P['items']; IT = lambda k, o, c: it[(it.item == k) & (it.ola == o)][c].iloc[0]
ld = P['psy']['Conjunto']['loadings']
Bw_ = R['by_wave']; IL = lambda: None
it7 = None
notas = [
 ('Índice de Problemas de Convivencia (IPC). Sección de convivencia de la EVPI (CC_1 a CC_9 sin CC_7), 2023-2024 vs. 2025-2026', 'h'), ('', ''),
 ('Construcción', 'b'),
 ('Ítems: la persona indica si en su barrio hay (1) peleas o riñas en el espacio público, (2) pandillas o grupos delictivos, (3) consumo de alcohol en el espacio público, (4) venta clandestina de alcohol, (5) consumo de drogas, (6) venta de drogas, (8) apropiación ilegal de viviendas por grupos delictivos y (9) reclutamiento de niñas, niños y adolescentes.', ''),
 ('Calles sin iluminación (CC_7) no forma parte del índice: es una condición de infraestructura del barrio, no un problema de convivencia, y se usa como variable explicativa.', ''),
 ('Cada ítem vale 1 si responde "sí" y 0 si responde "no". Ns/Nc queda como faltante. IPC = porcentaje de los problemas respondidos que la persona reporta (0-100), para quienes responden al menos 5 de los 8. Más alto = más problemas de convivencia.', ''),
 (f"Cobertura: {P['cover'][0]['con_indice']:,} de {P['cover'][0]['n']:,} casos en 2023-2024 y {P['cover'][1]['con_indice']:,} de {P['cover'][1]['n']:,} en 2025-2026.".replace(',', '.'), ''),
 ('', ''),
 ('Validez y confiabilidad', 'b'),
 (f"Un factor dominante: primer autovalor {es(P['psy']['Conjunto']['eig'][0], 2)}, segundo {es(P['psy']['Conjunto']['eig'][1], 2)}. α ordinal = {es(P['alpha_ord']['Conjunto'], 2)} (sube desde 0,91 al quitar iluminación); KR-20 = {es(P['kr20']['Conjunto'], 2)}. Congruencia de Tucker entre olas = {es(P['tucker'], 3)}.", ''),
 ('Subdimensiones: el desorden cotidiano (peleas, pandillas, alcohol y drogas) concentra las cargas más altas; apropiación de viviendas y reclutamiento de NNA forman un componente de presencia criminal grave, menos frecuente. Se reportan subíndices de ambos.', ''),
 (f"Funcionamiento diferencial entre olas: para un mismo nivel general, en 2025-2026 se reporta más consumo de drogas (OR {es(P['dif'].set_index('item').loc['CC_5', 'or_ola'], 2)}) y apropiación de viviendas (OR {es(P['dif'].set_index('item').loc['CC_8', 'or_ola'], 2)}), y menos venta de alcohol (OR {es(P['dif'].set_index('item').loc['CC_4', 'or_ola'], 2)}) y reclutamiento (OR {es(P['dif'].set_index('item').loc['CC_9', 'or_ola'], 2)}). Cambió la composición de los problemas, no solo su nivel.", ''),
 ('', ''),
 ('Hallazgos principales', 'b'),
 (f"1. Los problemas de convivencia aumentaron: el IPC sube de {es(c0.m23)} a {es(c0.m25)} (+{es(c0.dif)} puntos, p < 0,001). El subíndice de desorden cotidiano sube {es(ch.loc['desorden', 'dif'])} puntos.", ''),
 ('2. El mayor aumento es el consumo de drogas en el espacio público (46,4% → 54,5%), seguido de pandillas (+4,8 pp), venta de drogas (+4,3) y consumo de alcohol (+3,8). Bajan la venta clandestina de alcohol y el reclutamiento.', ''),
 (f"3. Calles sin iluminación es el factor más asociado al índice: suma {es(M4['params']['sin_ilum'])} puntos con todo lo demás constante. Su peso bajó entre olas ({es(Bw_.loc['sin_ilum', 'b_23'])} → {es(Bw_.loc['sin_ilum', 'b_25'])}, p < 0,001): los problemas crecieron en barrios iluminados (+{es(Sg('Calles sin iluminación', 'Sin problema de iluminación').dif)} puntos) y no en los que reportan calles oscuras (+{es(Sg('Calles sin iluminación', 'Con calles sin iluminación').dif)}, n.s.).", ''),
 (f"4. Otros factores: poco orgullo del barrio (+{es(M4['params']['poco_orgullo'])}), sentirse inseguro (+{es(M4['params']['inseg_barrio'])}), creer que la delincuencia aumentó (+{es(M4['params']['delinc_aumento'])}), haber sido víctima (+{es(M4['params']['vic_robo_viol'])} a +{es(M4['params']['vic_otro_pers'])}) y no confiar en la Policía (+{es(M4['params']['conf_pol_no'])}). Las personas de 65 años o más ({es(M4['params']['e65'])}) y de NSE alto ({es(M4['params']['nse_alto'])}) reportan menos. R² = {es(M4['r2'], 3)}.", ''),
 (f"5. Territorio: Tumbaco (+{es(Sg('Administración zonal', 'Tumbaco').dif)}), Manuela Sáenz (+{es(Sg('Administración zonal', 'Manuela Sáenz').dif)}), Choco Andino (+{es(Sg('Administración zonal', 'Choco Andino').dif)}) y Los Chillos (+{es(Sg('Administración zonal', 'Los Chillos').dif)}) empeoran; Quitumbe mejora ({es(Sg('Administración zonal', 'Quitumbe').dif)}). Las zonas que partían con menos problemas convergen hacia el promedio.", ''),
 (f"6. Entre las víctimas el IPC sube {es(Sg('Victimización personal (12 meses)', 'Víctima').dif)} puntos; entre las no víctimas no cambia ({es(Sg('Victimización personal (12 meses)', 'No víctima').dif)}).", ''),
 (f"7. La composición de la muestra explica {es(ob.loc['explicado', 'estimado'], 2)} de {es(ob.loc['total', 'estimado'], 2)} puntos (Oaxaca-Blinder). Los determinantes cambian entre olas (χ²({R['wald_noaz']['df']}) = {es(R['wald_noaz']['chi2'])}, p < 0,001): pesa menos la iluminación, desaparece el gradiente de ingreso y se fortalece la victimización.", ''),
 ('', ''),
 ('Especificación y límites', 'b'),
 ('Ponderación con peso_az_estrato. MCO ponderado con errores robustos HC1. Las covariables excluyen civiles armados, parte del fenómeno medido. Descomposición Oaxaca-Blinder con IC bootstrap (300 réplicas).', ''),
 (f"Ns/Nc subió en los ítems sensibles (venta de drogas 9,1% → 15,8%; reclutamiento 4,5% → 10,6%). Con Ns/Nc = no el aumento es {es(ch.loc['ipc_ns0', 'dif'])} puntos (p = 0,01); con Ns/Nc = sí, {es(ch.loc['ipc_ns1', 'dif'])}. Puede reflejar más temor a responder.", ''),
 ('Iluminación como explicativa: es una asociación. Los barrios con calles oscuras pueden compartir otras carencias (ingreso, infraestructura, presencia institucional) no observadas. Aun así, es una condición sobre la que el Municipio puede actuar directamente.', ''),
 ('Mide percepción de problemas en el barrio, no incidencia registrada.', ''),
 ('', ''),
 ('Fuente: EVPI 2023-2024 y 2025-2026, OMSCGR. Cálculos propios en Python (statsmodels, girth). *** p<0,001; ** p<0,01; * p<0,05; † p<0,10.', ''),
]
for i, (t, s) in enumerate(notas, 1):
    c = ws.cell(i, 1, t); c.alignment = Alignment(wrap_text=True, vertical='top'); c.font = fnt(bold=s in ('h', 'b'), size=13 if s == 'h' else 10)
ws.column_dimensions['A'].width = 150

# Ítems
ws = wb.create_sheet('Ítems'); title(ws, 'Problemas de convivencia reportados en el barrio, por ítem y ola', '% que responde sí entre quienes responden sí o no; % Ns/Nc sobre el total. Ponderado.')
head(ws, 4, ['Código', 'Problema', '% sí 2023', '% sí 2025', 'Cambio (pp)', '% Ns/Nc 2023', '% Ns/Nc 2025', 'n 2023', 'n 2025'])
for i, k in enumerate(ITEMS, 5):
    a = I[(I.item == k) & (I.ola == 0)].iloc[0]; b = I[(I.item == k) & (I.ola == 1)].iloc[0]
    put(ws, i, [k, ILAB[k], a.pct_si, b.pct_si, f'=(D{i}-C{i})*100', a.pct_ns, b.pct_ns, a.n, b.n],
        {3: '0.0%', 4: '0.0%', 5: '+0.0;-0.0', 6: '0.0%', 7: '0.0%', 8: '#,##0', 9: '#,##0'}, blue=(3, 4, 6, 7, 8, 9))
widths(ws, [8, 50, 11, 11, 11, 12, 12, 8, 8])

# Psicometría
ws = wb.create_sheet('Psicometría'); title(ws, 'Validez y confiabilidad del índice')
r = 3; ws.cell(r, 1, 'Cargas factoriales (1 factor, ejes principales sobre correlaciones tetracóricas ponderadas)').font = fnt(bold=True, size=10); r += 1
head(ws, r, ['Código', 'Problema', 'Conjunto', '2023-2024', '2025-2026', 'Correlación ítem-resto', 'IRT a (discriminación)', 'IRT b (dificultad)', 'a 2023', 'a 2025', 'b 2023', 'b 2025', 'α ordinal sin el ítem']); r += 1
for k in ITEMS:
    put(ws, r, [k, ILAB[k], P['psy']['Conjunto']['loadings'][k], P['psy']['2023-2024']['loadings'][k], P['psy']['2025-2026']['loadings'][k],
                P['item_rest'].get(k, None), P['irt'].loc[k, 'a'], P['irt'].loc[k, 'b'], P['irt_wave'][0].loc[k, 'a'], P['irt_wave'][1].loc[k, 'a'],
                P['irt_wave'][0].loc[k, 'b'], P['irt_wave'][1].loc[k, 'b'], P['alpha_if_deleted'][k]], {j: '0.000' for j in range(3, 14)}, blue=range(3, 14)); r += 1
r += 1
for lab_, v in [('Primer autovalor (conjunto)', P['psy']['Conjunto']['eig'][0]), ('Segundo autovalor (conjunto)', P['psy']['Conjunto']['eig'][1]),
                ('α ordinal conjunto', P['alpha_ord']['Conjunto']), ('α ordinal 2023-2024', P['alpha_ord']['2023-2024']), ('α ordinal 2025-2026', P['alpha_ord']['2025-2026']),
                ('KR-20 (8 ítems, casos completos)', P['kr20']['Conjunto']), ('Congruencia de Tucker entre olas', P['tucker'])]:
    ws.cell(r, 1, lab_).font = fnt(size=9); c = ws.cell(r, 3, v); c.number_format = '0.000'; c.font = fnt(size=9, color=BIN); r += 1
r += 1; ws.cell(r, 1, 'Funcionamiento diferencial por ola (DIF): logit del ítem sobre puntaje resto y ola').font = fnt(bold=True, size=10); r += 1
head(ws, r, ['Código', 'Problema', 'OR ola (DIF uniforme)', 'p uniforme', 'p no uniforme', 'n']); r += 1
for x in P['dif'].itertuples():
    put(ws, r, [x.item, ILAB[x.item], x.or_ola, x.p_unif, x.p_nounif, x.n], {3: '0.00', 4: '0.000', 5: '0.000', 6: '#,##0'}, blue=(3, 4, 5, 6)); r += 1
r += 1; ws.cell(r, 1, 'Correlaciones tetracóricas (conjunto)').font = fnt(bold=True, size=10); r += 1
head(ws, r, [''] + ITEMS); r += 1
T = P['psy']['Conjunto']['tetra']
for k in ITEMS:
    put(ws, r, [k] + list(T.loc[k].values), {j: '0.00' for j in range(2, 10)}, blue=range(2, 10)); r += 1
widths(ws, [10, 46, 11, 11, 11, 12, 12, 12, 9, 9, 9, 9, 12])

# Versiones y cambio
ws = wb.create_sheet('Índice y cambio'); title(ws, 'Cambio del índice entre olas, por versión', 'MCO ponderado (índice ~ ola) con EE robustos HC1, salvo indicación.')
head(ws, 4, ['Versión', 'Descripción', '2023-2024', '2025-2026', 'Diferencia', 'EE', 'IC95 inf', 'IC95 sup', 'p', 'n'])
for i, x in enumerate(R['change'].itertuples(), 5):
    fm = '0.000' if x.version in ('alguno', 'grave', 'criminal', 'ipc_irt') else ('0.00' if x.version == 'conteo' else '0.0')
    put(ws, i, [x.version, x.desc, x.m23, x.m25, f'=D{i}-C{i}', x.se, f'=E{i}-1.96*F{i}', f'=E{i}+1.96*F{i}', x.p, x.n],
        {3: fm, 4: fm, 5: '+' + fm + ';-' + fm, 6: fm, 7: '+' + fm + ';-' + fm, 8: '+' + fm + ';-' + fm, 9: '0.000', 10: '#,##0'}, blue=(3, 4, 6, 9, 10))
i += 2; ws.cell(i, 1, 'Correlaciones entre versiones').font = fnt(bold=True, size=10); i += 1
C = R['corr_idx'] if 'corr_idx' in R else P['corr_idx']
head(ws, i, [''] + list(C.columns)); i += 1
for k in C.index:
    put(ws, i, [k] + list(C.loc[k].values), {j: '0.000' for j in range(2, 7)}, blue=range(2, 7)); i += 1
i += 1; ws.cell(i, 1, 'Distribución del índice principal por tramos').font = fnt(bold=True, size=10); i += 1
head(ws, i, ['Tramo', '2023-2024', '2025-2026', 'Cambio (pp)']); i += 1
Dd = R['dist']
for t in list(dict.fromkeys(Dd.tramo)):
    a = Dd[(Dd.ola == 0) & (Dd.tramo == t)].pct.iloc[0]; b = Dd[(Dd.ola == 1) & (Dd.tramo == t)].pct.iloc[0]
    put(ws, i, [t, a, b, f'=(C{i}-B{i})*100'], {2: '0.0%', 3: '0.0%', 4: '+0.0;-0.0'}, blue=(2, 3)); i += 1
widths(ws, [14, 62, 11, 11, 11, 8, 10, 10, 8, 8])

# Subgrupos
ws = wb.create_sheet('Subgrupos'); title(ws, 'Índice de Problemas de Convivencia (0-100) por grupo y ola', 'Diferencia por MCO ponderado con EE robustos HC1.')
head(ws, 4, ['Grupo', 'Categoría', '2023-2024', '2025-2026', 'Diferencia', 'EE', 'IC95 inf', 'IC95 sup', 'p', 'Sig.', 'n'])
r = 5; prev = None
for x in R['subg'].itertuples():
    if x.grupo != prev:
        ws.cell(r, 1, x.grupo).font = fnt(bold=True, size=9)
        for j in range(1, 12): ws.cell(r, j).fill = BLK
        r += 1; prev = x.grupo
    put(ws, r, ['', x.categoria, x.m23, x.m25, f'=D{r}-C{r}', x.se, f'=E{r}-1.96*F{r}', f'=E{r}+1.96*F{r}', x.p, star(x.p), x.n],
        {3: '0.0', 4: '0.0', 5: '+0.0;-0.0', 6: '0.0', 7: '+0.0;-0.0', 8: '+0.0;-0.0', 9: '0.000', 11: '#,##0'}, blue=(3, 4, 6, 9, 11)); r += 1
widths(ws, [30, 24, 10, 10, 11, 7, 9, 9, 8, 6, 8]); ws.freeze_panes = 'C5'

# Modelos anidados
ws = wb.create_sheet('Modelos anidados'); title(ws, 'Determinantes del Índice de Problemas de Convivencia: coeficientes MCO ponderados (puntos; positivo = más problemas)', 'EE robustos HC1 entre paréntesis. *** p<0,001; ** p<0,01; * p<0,05; † p<0,10.')
names = list(R['models']); hs = ['Variable']
for nm in names: hs += [nm, '']
head(ws, 4, hs, 42)
for k in range(len(names)): ws.merge_cells(start_row=4, start_column=2 + 2*k, end_row=4, end_column=3 + 2*k)
r = 5
for b, vs in {'Ola': ['ola25'], **BLOQUES, 'Administración zonal (ref.: Eloy Alfaro)': AZD}.items():
    ws.cell(r, 1, b).font = fnt(bold=True, size=9)
    for j in range(1, 2 + 2*len(names)): ws.cell(r, j).fill = BLK
    r += 1
    for v in vs:
        ws.cell(r, 1, LAB[v]).font = fnt(size=9)
        for k, nm in enumerate(names):
            m = R['models'][nm]
            if v in m['params'].index:
                c = ws.cell(r, 2 + 2*k, m['params'][v]); c.number_format = '+0.00;-0.00'; c.font = fnt(size=9, bold=m['p'][v] < .05)
                c = ws.cell(r, 3 + 2*k, f"({m['bse'][v]:.2f}){star(m['p'][v])}".replace('.', ',')); c.font = fnt(size=8, color='555555')
        r += 1
for lab_, key, fm in [('Constante', 'const', '+0.00;-0.00'), ('Observaciones', 'n', '#,##0'), ('R²', 'r2', '0.000')]:
    ws.cell(r, 1, lab_).font = fnt(bold=True, size=9)
    for k, nm in enumerate(names):
        m = R['models'][nm]; v = m['params']['const'] if key == 'const' else m[key]
        c = ws.cell(r, 2 + 2*k, v); c.font = fnt(size=9); c.number_format = fm
    r += 1
widths(ws, [48] + [9, 12]*len(names)); ws.freeze_panes = 'B5'

# Por ola
ws = wb.create_sheet('Determinantes por ola'); title(ws, '¿Cambiaron los determinantes del índice entre olas?', 'MCO ponderado M4 por ola con EF de AZ. Diferencia = 2025 − 2023.')
head(ws, 4, ['Variable', 'b 2023-2024', 'EE', 'p', 'b 2025-2026', 'EE', 'p', 'Diferencia', 'EE dif.', 'z', 'p dif.', 'Sig. dif.'], 32)
r = 5
for b, vs in {**BLOQUES, 'Administración zonal (ref.: Eloy Alfaro)': AZD}.items():
    ws.cell(r, 1, b).font = fnt(bold=True, size=9)
    for j in range(1, 13): ws.cell(r, j).fill = BLK
    r += 1
    for v in vs:
        e = Bw.loc[v]
        put(ws, r, [LAB[v], e.b_23, e.se_23, e.p_23, e.b_25, e.se_25, e.p_25, f'=E{r}-B{r}', f'=SQRT(C{r}^2+F{r}^2)', f'=H{r}/I{r}',
                    f'=2*(1-NORMSDIST(ABS(J{r})))', f'=IF(K{r}<0.001,"***",IF(K{r}<0.01,"**",IF(K{r}<0.05,"*",IF(K{r}<0.1,"†",""))))'],
            {2: '+0.00;-0.00', 3: '0.00', 4: '0.000', 5: '+0.00;-0.00', 6: '0.00', 7: '0.000', 8: '+0.00;-0.00', 9: '0.00', 10: '0.00', 11: '0.000'}, blue=(2, 3, 4, 5, 6, 7))
        r += 1
ws.conditional_formatting.add(f'A5:L{r-1}', FormulaRule(formula=['AND(ISNUMBER($K5),$K5<0.05)'], font=Font(name=F, bold=True)))
r += 1
for nm, t in [('wald', 'todas las interacciones, incl. AZ'), ('wald_noaz', 'sin efectos fijos de AZ')]:
    ws.cell(r, 1, f"Prueba de Wald conjunta ({t}): χ²({R[nm]['df']}) = {R[nm]['chi2']:.1f}; p = {R[nm]['p']:.2e}").font = fnt(size=9, italic=True); r += 1
widths(ws, [48, 11, 7, 7, 11, 7, 7, 10, 7, 7, 8, 7]); ws.freeze_panes = 'B5'

# Descomposición
ws = wb.create_sheet('Descomposición'); title(ws, 'Descomposición Oaxaca-Blinder del cambio del índice (puntos)', 'Coeficientes del modelo conjunto con indicador de ola. IC 95% bootstrap (300 réplicas).')
head(ws, 4, ['Componente', 'Estimado', 'EE bootstrap', 'IC95 inf', 'IC95 sup', '% del cambio'])
labo = {'total': 'Cambio total (2025 − 2023)', 'explicado': 'Explicado por composición', 'no_explicado': 'No explicado'}
for i, (k, x) in enumerate(ob.iterrows(), 5):
    put(ws, i, [labo.get(k, '   ' + k.replace('E: ', 'Explicado: ')), x.estimado, x.se, x.lo, x.hi, f'=B{i}/$B$5'],
        {2: '+0.00;-0.00', 3: '0.00', 4: '+0.00;-0.00', 5: '+0.00;-0.00', 6: '0.0%'}, blue=(2, 3, 4, 5))
    if k in labo: ws.cell(i, 1).font = fnt(size=9, bold=True)
i += 2; o2 = R['oaxaca_alt']
ws.cell(i, 1, f"Subíndice de desorden cotidiano: total {es(o2['total'], 2)}; explicado {es(o2['explicado'], 2)}; no explicado {es(o2['no_explicado'], 2)}.").font = fnt(size=9, italic=True)
widths(ws, [52, 11, 13, 11, 11, 12])

# Sensibilidad
ws = wb.create_sheet('Sensibilidad'); title(ws, 'Efecto de la ola condicional a covariables (M4), según especificación')
head(ws, 4, ['Especificación', 'Coef. ola 2025', 'EE', 'IC95 inf', 'IC95 sup', 'p', 'n'])
for i, x in enumerate(R['sens'].itertuples(), 5):
    fm = '0.000' if 'IRT' in x.especificacion else '0.00'
    put(ws, i, [x.especificacion, x.ola, x.se, f'=B{i}-1.96*C{i}', f'=B{i}+1.96*C{i}', x.p, x.n],
        {2: '+' + fm + ';-' + fm, 3: fm, 4: '+' + fm + ';-' + fm, 5: '+' + fm + ';-' + fm, 6: '0.000', 7: '#,##0'}, blue=(2, 3, 6, 7))
ws.cell(i + 2, 1, 'Las versiones ordinal (0-100) e IRT (desviaciones estándar) están en otra escala que la principal.').font = fnt(size=8, italic=True)
widths(ws, [52, 13, 8, 10, 10, 8, 8])

# Figuras
ws = wb.create_sheet('Figuras'); title(ws, 'Figuras'); pos = 3
for f, h in [('ipc_fig1_items.png', 440), ('ipc_fig2_cambio.png', 470), ('ipc_fig3_determinantes.png', 900)]:
    img = XLImage(f'{SALIDA}/' + f); ratio = img.width / img.height; img.height = h; img.width = int(h * ratio)
    ws.add_image(img, f'A{pos}'); pos += int(h / 20) + 3

# Índice por persona
ws = wb.create_sheet('Índice por persona'); title(ws, 'Índice por persona (para unir a la base por ID_Todo)')
cols = ['ID_Todo', 'AÑO', 'ADMINISTRACION ZONAL', 'peso_az_estrato']
out = d[cols].copy()
for c, n in [('n_items', 'n_items'), ('ipc', 'ipc'), ('desorden', 'ipc_desorden'), ('criminal', 'presencia_criminal'), ('ipc_ns0', 'ipc_ns0'), ('ipc_ns1', 'ipc_ns1'), ('ipc_irt', 'ipc_irt'), ('conteo', 'conteo'), ('sin_ilum', 'calles_sin_iluminacion')]: out[n] = IDX[c]
head(ws, 3, list(out.columns))
for i, row in enumerate(out.itertuples(index=False), 4):
    for j, v in enumerate(row, 1):
        ws.cell(i, j, None if (isinstance(v, float) and np.isnan(v)) else v)
widths(ws, [12, 7, 20, 14, 8, 8, 11, 11, 9, 9, 9, 8, 12]); ws.freeze_panes = 'A4'
wb.save(f'{SALIDA}/EVPI_indice_problemas_convivencia.xlsx'); print('Guardado en', SALIDA)

## 10. Descargar resultados

In [ ]:
import shutil
shutil.make_archive('salidas_ipc', 'zip', SALIDA)
if EN_COLAB:
    files.download(f'{SALIDA}/EVPI_indice_problemas_convivencia.xlsx')
    # files.download('salidas_ipc.zip')   # Excel + figuras